# 01 - Data Cleaning & Preparation
### US Accidents (2016–2023) — Phân tích tai nạn giao thông đô thị

**Thành viên phụ trách:** Nguyễn Trọng Quý — MSSV: 52400230 — Vai trò: Data Engineer
**Branch:** `member/quy-52400230`

**Mục tiêu của notebook này:**
1. Nạp dữ liệu gốc US Accidents từ Kaggle (qua Kaggle API).
2. Lọc dữ liệu để chỉ giữ lại **2 năm gần nhất** theo yêu cầu của Sở GTVT — dùng logic **động** (dynamic), không hard-code năm cụ thể, để đảm bảo khả năng tái lập (reproducibility) khi dataset được cập nhật.
3. Xử lý dữ liệu thiếu (missing values) theo **chiến lược phân tầng (tiered strategy)** dựa trên tỉ lệ thiếu của từng cột.
4. Chuẩn hóa định dạng thời gian và trích xuất các đặc trưng thời gian: `Hour`, `DayOfWeek`, `Month`, `Season`.
5. Thực hiện **time-series train/test split** (chia theo thời gian) thay vì chia ngẫu nhiên, để tránh rò rỉ dữ liệu (data leakage) — vì đây là dữ liệu chuỗi thời gian.
6. Xuất dữ liệu đã làm sạch cho Member 3 (visualization) và Member 4 (modeling) sử dụng.

> **Lưu ý quan trọng về Git:** File dữ liệu thô (CSV, ~2-3GB) **không được commit** lên repo. Xem mục cuối notebook để biết cấu hình `.gitignore` và script tải dữ liệu kèm theo (`download_data.py`).


## 1. Import thư viện

In [3]:
import os
import numpy as np
import pandas as pd
import kagglehub
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 2. Nạp dữ liệu gốc từ Kaggle bằng KaggleHub

Dataset sử dụng: **US Accidents (2016-2023) by Sobhan Moosavi**.

Notebook này sử dụng `kagglehub.dataset_download()` để tự động tải dataset về máy. Cách này thay thế cho việc yêu cầu người dùng tự tải CSV và đặt thủ công vào `data/raw`.

Sau khi tải, notebook sẽ:
1. In ra thư mục dataset mà KaggleHub trả về.
2. Liệt kê các file trong thư mục.
3. Tìm các file `.csv`.
4. Chọn file CSV phù hợp và đọc vào DataFrame.


In [4]:
# ==========================================
# 1. Tải dataset từ Kaggle bằng KaggleHub
# ==========================================
dataset_path = kagglehub.dataset_download("sobhanmoosavi/us-accidents")

print("Dataset location:")
print(dataset_path)

# ==========================================
# 2. Liệt kê các file trong dataset
# ==========================================
files = os.listdir(dataset_path)

print("\nFiles in dataset:")
for file in files:
    print(file)

# ==========================================
# 3. Tìm file CSV
# ==========================================
csv_files = [file for file in files if file.lower().endswith(".csv")]

print("\nCSV files:")
for file in csv_files:
    print(file)

if not csv_files:
    raise FileNotFoundError("Khong tim thay file CSV trong dataset Kaggle.")

# Nếu có nhiều CSV, chọn file lớn nhất vì đây thường là file dữ liệu chính.
csv_path = max(
    [os.path.join(dataset_path, file) for file in csv_files],
    key=os.path.getsize
)

print("\nLoading:")
print(csv_path)

# ==========================================
# 4. Đọc các cột cần thiết
# ==========================================
USECOLS = [
    "ID", "Severity", "Start_Time", "End_Time",
    "Start_Lat", "Start_Lng", "City", "County", "State", "Zipcode",
    "Temperature(F)", "Humidity(%)", "Pressure(in)", "Visibility(mi)",
    "Wind_Speed(mph)", "Precipitation(in)", "Weather_Condition",
    "Amenity", "Crossing", "Junction", "Railway", "Stop", "Traffic_Signal",
    "Sunrise_Sunset",
]

# Kiểm tra tên cột trước khi đọc để tránh lỗi nếu Kaggle cập nhật dataset.
available_columns = pd.read_csv(csv_path, nrows=0).columns.tolist()
missing_required_columns = [col for col in USECOLS if col not in available_columns]

if missing_required_columns:
    print("\nCanh bao - cac cot khong ton tai:")
    print(missing_required_columns)

USECOLS_EXISTING = [col for col in USECOLS if col in available_columns]

df_raw = pd.read_csv(
    csv_path,
    usecols=USECOLS_EXISTING,
    low_memory=False
)

print("\nDataset shape:")
print(df_raw.shape)

print("\nFirst 5 records:")
df_raw.head()


Dataset location:
C:\Users\User\.cache\kagglehub\datasets\sobhanmoosavi\us-accidents\versions\13

Files in dataset:
US_Accidents_March23.csv

CSV files:
US_Accidents_March23.csv

Loading:
C:\Users\User\.cache\kagglehub\datasets\sobhanmoosavi\us-accidents\versions\13\US_Accidents_March23.csv

Dataset shape:
(7728394, 24)

First 5 records:


,ID,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,City,County,State,Zipcode,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Crossing,Junction,Railway,Stop,Traffic_Signal,Sunrise_Sunset
0,A-1,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,Dayton,Montgomery,OH,45424,36.9,91.0,29.68,10.0,NaN,0.02,Light Rain,False,False,False,False,False,False,Night
1,A-2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.928059,-82.831184,Reynoldsburg,Franklin,OH,43068-3402,37.9,100.0,29.65,10.0,NaN,0.00,Light Rain,False,False,False,False,False,False,Night
2,A-3,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.063148,-84.032608,Williamsburg,Clermont,OH,45176,36.0,100.0,29.67,10.0,3.5,NaN,Overcast,False,False,False,False,False,True,Night
3,A-4,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.747753,-84.205582,Dayton,Montgomery,OH,45417,35.1,96.0,29.64,9.0,4.6,NaN,Mostly Cloudy,False,False,False,False,False,False,Night
4,A-5,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.627781,-84.188354,Dayton,Montgomery,OH,45459,36.0,89.0,29.65,6.0,3.5,NaN,Mostly Cloudy,False,False,False,False,False,True,Day


## 3. Lọc dữ liệu — 2 năm gần nhất (dynamic date logic)

**Vì sao dùng logic động thay vì hard-code năm?**
Nếu hard-code (ví dụ `df[df.Year >= 2022]`), notebook sẽ **sai** khi Kaggle cập nhật dataset
với dữ liệu mới hơn, hoặc khi ai đó chạy lại pipeline này ở thời điểm khác.
Thay vào đó, ta luôn lấy **mốc thời gian mới nhất có trong chính dataset** (`Start_Time.max()`)
làm điểm neo, rồi lùi lại đúng 2 năm — đảm bảo pipeline luôn tái lập được (reproducible)
bất kể dataset được tải về vào lúc nào.


In [5]:
# Chuẩn hóa các cột thời gian về datetime.
df_raw["Start_Time"] = pd.to_datetime(
    df_raw["Start_Time"], errors="coerce", format="mixed"
)
df_raw["End_Time"] = pd.to_datetime(
    df_raw["End_Time"], errors="coerce", format="mixed"
)

# Loại các dòng không parse được Start_Time.
n_before = len(df_raw)
df_raw = df_raw.dropna(subset=["Start_Time"]).copy()
print(f"Loai {n_before - len(df_raw):,} dong do Start_Time khong hop le.")

# Dynamic date filtering:
# lấy mốc thời gian mới nhất thực sự có trong dataset rồi lùi lại 2 năm.
latest_date = df_raw["Start_Time"].max()
cutoff_date = latest_date - pd.DateOffset(years=2)

print(f"Moc thoi gian gan nhat trong dataset : {latest_date}")
print(f"Cutoff (2 nam truoc moc gan nhat)     : {cutoff_date}")

df_recent = df_raw[df_raw["Start_Time"] >= cutoff_date].copy()

print(
    f"So dong sau khi loc 2 nam gan nhat    : {len(df_recent):,} / "
    f"{len(df_raw):,} ({len(df_recent) / len(df_raw):.1%})"
)

df_recent.sort_values("Start_Time", inplace=True)
df_recent.reset_index(drop=True, inplace=True)

print("\nKhoang thoi gian sau khi loc:")
print(df_recent["Start_Time"].min(), "->", df_recent["Start_Time"].max())


Loai 0 dong do Start_Time khong hop le.
Moc thoi gian gan nhat trong dataset : 2023-03-31 23:30:00
Cutoff (2 nam truoc moc gan nhat)     : 2021-03-31 23:30:00
So dong sau khi loc 2 nam gan nhat    : 3,174,360 / 7,728,394 (41.1%)

Khoang thoi gian sau khi loc:
2021-03-31 23:37:00 -> 2023-03-31 23:30:00


## 4. Xử lý dữ liệu thiếu — chiến lược phân tầng (tiered strategy)

Thay vì áp dụng một cách xử lý duy nhất cho toàn bộ dataset, ta phân loại các cột theo
**tỉ lệ thiếu (% missing)** rồi áp dụng chiến lược phù hợp cho từng nhóm:

| Tỉ lệ thiếu | Chiến lược | Lý do |
|---|---|---|
| **> 40%** | Loại bỏ cột | Quá nhiều thông tin bị mất, impute sẽ gây nhiễu/sai lệch lớn |
| **5% – 40%** | Impute (median cho numeric, mode cho categorical) | Đủ dữ liệu để ước lượng hợp lý, không nên bỏ vì mất quá nhiều dòng |
| **< 5%** | Loại bỏ dòng (row-wise drop) | Tỉ lệ nhỏ, drop dòng không ảnh hưởng đáng kể đến kích thước mẫu |

Ngưỡng (40% / 5%) có thể điều chỉnh tùy đặc thù dữ liệu — được ghi rõ trong biến `HIGH_MISSING_THRESHOLD`
và `LOW_MISSING_THRESHOLD` bên dưới để dễ tinh chỉnh và giải trình trong báo cáo.


In [7]:
HIGH_MISSING_THRESHOLD = 0.40   # > 40% missing -> drop column
LOW_MISSING_THRESHOLD = 0.05    # < 5% missing  -> drop rows

missing_pct = df_recent.isna().mean().sort_values(ascending=False)
missing_report = missing_pct[missing_pct > 0].to_frame("missing_pct")
missing_report["missing_pct"] = (missing_report["missing_pct"] * 100).round(2)
print("Bao cao ty le thieu du lieu theo cot:")
missing_report


Bao cao ty le thieu du lieu theo cot:


,missing_pct
Precipitation(in),3.86
Wind_Speed(mph),2.64
Visibility(mi),2.26
Humidity(%),2.25
Weather_Condition,2.17
Temperature(F),2.13
Pressure(in),1.85
Sunrise_Sunset,0.68
Zipcode,0.02
City,0.00


In [8]:
cols_high_missing = missing_pct[missing_pct > HIGH_MISSING_THRESHOLD].index.tolist()
cols_mid_missing = missing_pct[
    (missing_pct <= HIGH_MISSING_THRESHOLD) &
    (missing_pct >= LOW_MISSING_THRESHOLD)
].index.tolist()
cols_low_missing = missing_pct[
    (missing_pct < LOW_MISSING_THRESHOLD) &
    (missing_pct > 0)
].index.tolist()

print("Cot loai bo (>40% missing)      :", cols_high_missing)
print("Cot impute (5%-40% missing)     :", cols_mid_missing)
print("Cot drop rows (<5% missing)     :", cols_low_missing)

# Tier 1: loại bỏ các cột có tỷ lệ thiếu quá cao.
df_clean = df_recent.drop(columns=cols_high_missing).copy()

# Tier 2: impute.
# Numeric -> median; categorical/object -> mode.
for col in cols_mid_missing:
    if col not in df_clean.columns:
        continue

    if pd.api.types.is_numeric_dtype(df_clean[col]):
        fill_value = df_clean[col].median()
    else:
        mode_values = df_clean[col].mode(dropna=True)
        fill_value = mode_values.iloc[0] if not mode_values.empty else "Unknown"

    df_clean[col] = df_clean[col].fillna(fill_value)

# Tier 3: với các cột có <5% missing, loại bỏ dòng còn thiếu.
low_missing_existing = [c for c in cols_low_missing if c in df_clean.columns]
if low_missing_existing:
    df_clean.dropna(subset=low_missing_existing, inplace=True)

print(f"\nKich thuoc truoc xu ly missing: {df_recent.shape}")
print(f"Kich thuoc sau xu ly missing  : {df_clean.shape}")
print(f"So gia tri missing con lai    : {int(df_clean.isna().sum().sum()):,}")


Cot loai bo (>40% missing)      : []
Cot impute (5%-40% missing)     : []
Cot drop rows (<5% missing)     : ['Precipitation(in)', 'Wind_Speed(mph)', 'Visibility(mi)', 'Humidity(%)', 'Weather_Condition', 'Temperature(F)', 'Pressure(in)', 'Sunrise_Sunset', 'Zipcode', 'City']

Kich thuoc truoc xu ly missing: (3174360, 24)
Kich thuoc sau xu ly missing  : (2972890, 24)
So gia tri missing con lai    : 0


### 5.1. Feature Engineering — Đặc trưng thời gian

Sau khi đã làm sạch dữ liệu và xử lý missing values ở mục 4, bước tiếp theo là
**trích xuất các đặc trưng thời gian** từ cột `Start_Time` gốc. Đây là bước quan
trọng vì hầu hết các mô hình học máy đều **không thể học trực tiếp** từ một cột
`datetime` dạng thô — mà cần các đặc trưng số và phân loại tường minh.

Các đặc trưng thời gian được tạo ra sẽ phục vụ cho cả hai mục tiêu:

- **Phân tích mô tả / chẩn đoán (Member 3):** trả lời các câu hỏi như
  *"Tai nạn xảy ra nhiều nhất vào giờ nào?"*, *"Cuối tuần hay ngày thường nguy hiểm hơn?"*,
  *"Mùa nào có tần suất tai nạn cao nhất?"* — những câu hỏi cốt lõi của bài toán
  phân tích điểm đen tai nạn giao thông đô thị.
- **Mô hình hóa (Member 4):** cung cấp các biến đầu vào dạng số và phân loại
  cho các thuật toán học máy (Logistic Regression, Random Forest, XGBoost...).

#### 5.1.1. Danh sách các đặc trưng thời gian được tạo

| Đặc trưng | Kiểu dữ liệu | Nguồn | Mô tả | Giá trị |
|---|---|---|---|---|
| `Hour` | int (0–23) | `Start_Time.dt.hour` | Giờ xảy ra tai nạn (24h) | 0, 1, ..., 23 |
| `DayOfWeek` | category | `Start_Time.dt.day_name()` | Thứ trong tuần | Monday → Sunday |
| `Month` | int (1–12) | `Start_Time.dt.month` | Tháng xảy ra tai nạn | 1 → 12 |
| `Year` | int | `Start_Time.dt.year` | Năm xảy ra tai nạn | 2022, 2023 |
| `Season` | category | Suy ra từ `Month` | Mùa theo chuẩn khí tượng | Winter / Spring / Summer / Fall |

#### 5.1.2. Quy tắc phân chia mùa (Season)

Mùa được chia theo **chuẩn khí tượng học** (meteorological seasons), khác với
chuẩn thiên văn (astronomical seasons) — vì trong phân tích giao thông, người ta
quan tâm đến đặc điểm khí hậu theo tháng hơn là vị trí Trái Đất so với Mặt Trời:

| Mùa | Tháng | Đặc điểm giao thông |
|---|---|---|
| **Winter** | 12, 1, 2 | Đường trơn, tuyết rơi, tầm nhìn kém |
| **Spring** | 3, 4, 5 | Mưa nhiều, giao thông tăng sau mùa đông |
| **Summer** | 6, 7, 8 | Cao điểm du lịch, tai nạn do tốc độ cao |
| **Fall** | 9, 10, 11 | Lá rụng che khuất tầm nhìn, sương mù sớm |

#### 5.1.3. Vì sao cần tạo các đặc trưng này?

**1. Cột `Start_Time` gốc không dùng trực tiếp được cho mô hình**

Các thuật toán học máy (Random Forest, XGBoost, Logistic Regression...) không
hiểu được khái niệm "ngày 15 tháng 6 năm 2022". Chúng cần các con số cụ thể
như `Hour = 15`, `Month = 6`, `DayOfWeek = "Wednesday"`. Do đó ta phải
**phân rã (decompose)** cột datetime thành các thành phần ngữ nghĩa.

**2. Tính chu kỳ của thời gian**

Thời gian có tính **chu kỳ tự nhiên** mà mô hình cần nắm bắt:

- `Hour` giúp phát hiện **giờ cao điểm** (7–9h sáng, 17–19h chiều);
- `DayOfWeek` giúp phát hiện sự khác biệt giữa **ngày thường vs cuối tuần**;
- `Month` và `Season` giúp phát hiện **xu hướng theo mùa** (mùa mưa, mùa tuyết).

**3. Phục vụ phân tích điểm đen tai nạn**

Trong bài toán "Phân tích điểm đen tai nạn giao thông đô thị", việc biết tai nạn
xảy ra **vào lúc nào** quan trọng không kém việc biết nó xảy ra **ở đâu**. Ví dụ:

- Một giao lộ có thể là "điểm đen" vào **giờ tan tầm** (17–18h) nhưng bình thường vào các giờ khác;
- Một đoạn đường có thể nguy hiểm vào **mùa mưa** nhưng an toàn vào mùa khô.

#### 5.1.4. Kết quả sau bước 5.1

Sau khi chạy cell code tương ứng, `df_clean` sẽ có thêm **5 cột** mới:


In [15]:
# ============================================================
# MỤC 5.1 — FEATURE ENGINEERING: ĐẶC TRƯNG THỜI GIAN
# ============================================================
# Trích xuất từ Start_Time các thành phần thời gian phục vụ
# phân tích mô tả (Member 3) và mô hình hóa (Member 4).
# ============================================================

# ---------- 0. Kiểm tra tiền đề ----------
assert "df_clean" in globals(), (
    "Chưa có df_clean. Hãy chạy lại các mục 2–4 trước."
)
assert "Start_Time" in df_clean.columns, (
    "df_clean thiếu cột Start_Time."
)
assert pd.api.types.is_datetime64_any_dtype(df_clean["Start_Time"]), (
    "Start_Time chưa phải kiểu datetime. Kiểm tra lại mục 3."
)

# ---------- 1. Trích xuất các thành phần thời gian ----------
df_clean["Hour"]      = df_clean["Start_Time"].dt.hour
df_clean["DayOfWeek"] = df_clean["Start_Time"].dt.day_name()   # Monday, ...
df_clean["Month"]     = df_clean["Start_Time"].dt.month
df_clean["Year"]      = df_clean["Start_Time"].dt.year

# ---------- 2. Suy ra mùa theo chuẩn khí tượng ----------
def month_to_season(month: int) -> str:
    """Chia mùa theo chuẩn meteorological seasons."""
    if month in (12, 1, 2):
        return "Winter"
    elif month in (3, 4, 5):
        return "Spring"
    elif month in (6, 7, 8):
        return "Summer"
    else:  # 9, 10, 11
        return "Fall"

df_clean["Season"] = df_clean["Month"].apply(month_to_season)

# ---------- 3. Kiểm tra kết quả ----------
time_feature_cols = ["Hour", "DayOfWeek", "Month", "Year", "Season"]

print("Các đặc trưng thời gian đã tạo:")
for col in time_feature_cols:
    print(f"  - {col:12s} | dtype={str(df_clean[col].dtype):10s} | "
          f"n_unique={df_clean[col].nunique():3d} | "
          f"missing={df_clean[col].isna().sum():,}")

# Assert không có missing trong các đặc trưng mới
for col in time_feature_cols:
    assert df_clean[col].isna().sum() == 0, f"Cột {col} còn NaN."

# Assert miền giá trị hợp lệ
assert df_clean["Hour"].between(0, 23).all(),      "Hour ngoài [0, 23]"
assert df_clean["Month"].between(1, 12).all(),     "Month ngoài [1, 12]"
assert set(df_clean["Season"].unique()) <= {"Winter", "Spring", "Summer", "Fall"}, \
    "Season có giá trị lạ."

print(f"\nTổng số cột hiện tại: {len(df_clean.columns)}")

# ---------- 4. Xem mẫu 5 dòng ----------
preview_cols = ["Start_Time", "Hour", "DayOfWeek", "Month", "Season", "Year"]
df_clean[preview_cols].head()

Các đặc trưng thời gian đã tạo:
  - Hour         | dtype=int32      | n_unique= 24 | missing=0
  - DayOfWeek    | dtype=str        | n_unique=  7 | missing=0
  - Month        | dtype=int32      | n_unique= 12 | missing=0
  - Year         | dtype=int32      | n_unique=  3 | missing=0
  - Season       | dtype=str        | n_unique=  4 | missing=0

Tổng số cột hiện tại: 36


,Start_Time,Hour,DayOfWeek,Month,Season,Year
0,2021-03-31 23:37:00,23,Wednesday,3,Spring,2021
1,2021-03-31 23:37:00,23,Wednesday,3,Spring,2021
2,2021-03-31 23:37:30,23,Wednesday,3,Spring,2021
3,2021-03-31 23:37:30,23,Wednesday,3,Spring,2021
4,2021-03-31 23:37:30,23,Wednesday,3,Spring,2021


### 5.2. Feature Engineering — Đặc trưng thời tiết và đường bộ

Sau khi đã trích xuất các đặc trưng thời gian ở mục 5.1, bước tiếp theo là
**xây dựng các đặc trưng phái sinh từ thời tiết và hạ tầng đường bộ** — hai nhóm
yếu tố được chứng minh có ảnh hưởng trực tiếp đến tần suất và mức độ nghiêm trọng
của tai nạn giao thông. Đây là bước đặc biệt quan trọng đối với bài toán
**phân tích điểm đen tai nạn giao thông đô thị**, vì tai nạn không chỉ phụ thuộc
vào *thời điểm* mà còn phụ thuộc vào *điều kiện môi trường* và *hạ tầng đường bộ*.

Các đặc trưng thời tiết và đường bộ được tạo ra sẽ phục vụ cho cả hai mục tiêu:

- **Phân tích mô tả / chẩn đoán (Member 3):** trả lời các câu hỏi như
  *"Tai nạn xảy ra nhiều hơn khi trời mưa hay trời trong?"*,
  *"Tầm nhìn thấp có làm tăng mức độ nghiêm trọng không?"*,
  *"Các giao lộ có đèn giao thông có an toàn hơn không?"* — những câu hỏi
  cốt lõi để xác định **điểm đen tai nạn** và **nguyên nhân gốc**.
- **Mô hình hóa (Member 4):** cung cấp các biến đầu vào dạng phân loại gọn nhẹ
  (thay vì 101 giá trị thô của `Weather_Condition`) và các biến nhị phân
  dễ diễn giải (`Is_Adverse_Weather`, `Is_Low_Visibility`...).

#### 5.2.1. Danh sách các đặc trưng thời tiết được tạo

| Đặc trưng | Kiểu dữ liệu | Nguồn | Mô tả | Giá trị |
|---|---|---|---|---|
| `Is_Precipitation` | bool | `Precipitation(in) > 0` | Có mưa/tuyết tại thời điểm tai nạn | True / False |
| `Weather_Group` | category | Gom nhóm từ `Weather_Condition` | Nhóm thời tiết ngữ nghĩa (7 nhóm) | Clear / Cloudy / Rain / Snow / Fog / Storm / Other |
| `Visibility_Level` | category | Từ `Visibility(mi)` | Mức tầm nhìn (dặm) | Very_Low / Low / Medium / High |
| `Temp_Level` | category | Từ `Temperature(F)` | Mức nhiệt độ (°F) | Freezing / Cold / Mild / Hot |
| `Is_Adverse_Weather` | bool | Từ `Weather_Group` | Thời tiết xấu (Rain/Snow/Fog/Storm) | True / False |
| `Is_Low_Visibility` | bool | `Visibility(mi) < 1.0` | Tầm nhìn thấp dưới 1 dặm | True / False |
| `Is_Daytime` | bool | Từ `Sunrise_Sunset` | Ban ngày hay ban đêm | True / False |

#### 5.2.2. Quy tắc gom nhóm thời tiết (Weather_Group)

Cột `Weather_Condition` trong dữ liệu gốc có khoảng **101 giá trị khác nhau**
(Light Rain, Heavy Rain, Snow, Fog, Fair, Clear, Overcast, Thunderstorm...).
Nếu đưa trực tiếp vào mô hình dưới dạng one-hot encoding sẽ tạo ra hơn 100 cột mới,
gây **loãng không gian đặc trưng** (*curse of dimensionality*) và khiến mô hình
khó học tín hiệu chính. Do đó ta **gom 101 giá trị thành 7 nhóm ngữ nghĩa lớn**
theo thứ tự ưu tiên từ nguy hiểm nhất đến ít nguy hiểm nhất:

| Nhóm | Từ khóa nhận diện | Ý nghĩa giao thông |
|---|---|---|
| **Snow** | snow, sleet, ice, wintry, blizzard | Nguy hiểm nhất — đường trơn, tầm nhìn kém |
| **Storm** | thunder, storm, tornado | Gió mạnh, mưa lớn, nguy cơ cao |
| **Fog** | fog, mist, haze, smoke | Tầm nhìn giảm mạnh |
| **Rain** | rain, drizzle, shower | Đường trơn, phanh kém hiệu quả |
| **Cloudy** | cloud, overcast | Ít ảnh hưởng, chỉ giảm tầm nhìn nhẹ |
| **Clear** | clear, fair | Điều kiện lý tưởng |
| **Other** | (còn lại) | Không xác định rõ |

> **Lưu ý về thứ tự ưu tiên:** Trong dữ liệu có các giá trị hỗn hợp như
> `"Light Rain And Snow"` — cần ưu tiên Snow trước Rain vì điều kiện nguy hiểm hơn.
> Thứ tự ưu tiên được áp dụng trong code là:
> **Snow > Storm > Fog > Rain > Cloudy > Clear**.

#### 5.2.3. Quy tắc phân mức tầm nhìn và nhiệt độ

**Mức tầm nhìn (`Visibility_Level`)** — dựa trên ngưỡng an toàn giao thông:

| Mức | Khoảng (dặm) | Ý nghĩa |
|---|---|---|
| **Very_Low** | < 1 | Cực kỳ nguy hiểm — gần như không thấy đường |
| **Low** | 1 – 3 | Nguy hiểm — khó quan sát xe phía trước |
| **Medium** | 3 – 6 | Trung bình — cần chú ý |
| **High** | >= 6 | Tốt — tầm nhìn bình thường |

**Mức nhiệt độ (`Temp_Level`)** — dựa trên ngưỡng đóng băng và ảnh hưởng sinh lý:

| Mức | Khoảng (°F) | Ý nghĩa |
|---|---|---|
| **Freezing** | < 32 | Nguy cơ đóng băng mặt đường |
| **Cold** | 32 – 50 | Lạnh — cần chú ý khi phanh |
| **Mild** | 50 – 70 | Dễ chịu — điều kiện lý tưởng |
| **Hot** | >= 70 | Nóng — có thể ảnh hưởng tập trung lái xe |

#### 5.2.4. Đặc trưng đường bộ — Điểm quan tâm (POI)

Bộ dữ liệu US Accidents cung cấp **13 trường boolean** mô tả sự hiện diện của
các điểm quan tâm (Point of Interest — POI) gần vị trí tai nạn. Những đặc trưng
này **đặc biệt quan trọng** đối với bài toán phân tích điểm đen tai nạn đô thị:

| Cột POI | Ý nghĩa |
|---|---|
| `Amenity` | Gần tiện ích công cộng (trạm xăng, nhà hàng, bãi đỗ xe...) |
| `Bump` | Có gờ giảm tốc |
| `Crossing` | Có vạch / lối qua đường cho người đi bộ |
| `Give_Way` | Có biển nhường đường |
| `Junction` | Gần giao lộ |
| `No_Exit` | Đường cụt |
| `Railway` | Gần đường sắt |
| `Roundabout` | Có vòng xuyến |
| `Station` | Gần ga / trạm |
| `Stop` | Có biển dừng |
| `Traffic_Calming` | Có biện pháp làm dịu giao thông |
| `Traffic_Signal` | Có đèn tín hiệu giao thông |
| `Turning_Loop` | Có vòng xoay chuyển hướng |

Các cột này được **giữ nguyên và ép về kiểu `bool`** để đảm bảo tính nhất quán
khi phân tích và mô hình hóa.

#### 5.2.5. Vì sao cần tạo các đặc trưng này?

**1. Giảm chiều dữ liệu phân loại**

`Weather_Condition` có 101 giá trị → one-hot sẽ tạo 101 cột. Sau khi gom nhóm
còn **7 giá trị** → chỉ tạo 7 cột. Tiết kiệm 94 cột, giúp mô hình hội tụ nhanh
hơn và giảm nguy cơ overfitting.

**2. Tạo biến nhị phân dễ diễn giải**

Các biến như `Is_Adverse_Weather`, `Is_Low_Visibility` rất trực quan khi trình
bày trong báo cáo — có thể mô tả bằng câu: *"23% tai nạn xảy ra trong điều kiện
thời tiết xấu"* thay vì phải liệt kê 101 giá trị thời tiết.

**3. Phục vụ phân tích điểm đen tai nạn**

Việc biết tai nạn xảy ra **trong điều kiện nào** và **gần loại hạ tầng nào**
giúp xác định **nguyên nhân gốc** và đề xuất **giải pháp phù hợp**:

- Nếu nhiều tai nạn xảy ra gần `Junction` khi trời mưa → cần cải tạo giao lộ, thêm đèn;
- Nếu nhiều tai nạn xảy ra gần `Crossing` vào ban đêm → cần chiếu sáng, biển báo phản quang;
- Nếu nhiều tai nạn xảy ra trên `Railway` khi `Is_Low_Visibility` → cần barie tự động.

#### 5.2.6. Kết quả sau bước 5.2

Sau khi chạy cell code tương ứng, `df_clean` sẽ có thêm **7 cột thời tiết**
và **13 cột POI** (đã có sẵn từ đầu nhưng được ép kiểu):


In [16]:
# ============================================================
# MỤC 5.2 — FEATURE ENGINEERING: ĐẶC TRƯNG THỜI TIẾT VÀ ĐƯỜNG BỘ
# ============================================================
# Xây dựng các đặc trưng phái sinh từ thời tiết và hạ tầng đường bộ
# phục vụ phân tích mô tả (Member 3) và mô hình hóa (Member 4).
# Căn cứ: Data dictionary của Kaggle US Accidents (2016–2023).
# ============================================================

# ---------- 0. Kiểm tra tiền đề ----------
assert "df_clean" in globals(), (
    "Chưa có df_clean. Hãy chạy lại các mục 2–5.1 trước."
)
# Đảm bảo các cột weather gốc tồn tại
required_weather_cols = ["Precipitation(in)", "Weather_Condition",
                         "Visibility(mi)", "Temperature(F)"]
for col in required_weather_cols:
    assert col in df_clean.columns, (
        f"df_clean thiếu cột {col}. Kiểm tra lại USECOLS ở Mục 2."
    )

# ---------- 1. Đặc trưng khoảng thời gian trong ngày ----------
# 1.1 Ban ngày / ban đêm từ Sunrise_Sunset
if "Sunrise_Sunset" in df_clean.columns:
    df_clean["Is_Daytime"] = (
        df_clean["Sunrise_Sunset"].astype(str).str.strip().str.lower() == "day"
    )
else:
    print("[CẢNH BÁO] Không tìm thấy Sunrise_Sunset, bỏ qua Is_Daytime.")

# ---------- 2. Đặc trưng phái sinh từ thời tiết ----------
# 2.1 Có mưa/tuyết hay không
df_clean["Is_Precipitation"] = df_clean["Precipitation(in)"] > 0

# 2.2 Nhóm ngữ nghĩa thời tiết (101 → 7)
def classify_weather(cond) -> str:
    """Gom ~101 giá trị Weather_Condition thành 7 nhóm lớn có ý nghĩa nghiệp vụ."""
    if not isinstance(cond, str):
        return "Other"
    c = cond.lower()
    # Ưu tiên: tuyết > bão > sương mù > mưa > nhiều mây > trong
    if any(k in c for k in ["snow", "sleet", "ice", "wintry", "blizzard"]):
        return "Snow"
    if any(k in c for k in ["thunder", "storm", "tornado"]):
        return "Storm"
    if any(k in c for k in ["fog", "mist", "haze", "smoke"]):
        return "Fog"
    if any(k in c for k in ["rain", "drizzle", "shower"]):
        return "Rain"
    if any(k in c for k in ["cloud", "overcast"]):
        return "Cloudy"
    if any(k in c for k in ["clear", "fair"]):
        return "Clear"
    return "Other"

df_clean["Weather_Group"] = df_clean["Weather_Condition"].apply(classify_weather)

# 2.3 Mức tầm nhìn
def classify_visibility(v) -> str:
    if pd.isna(v):  return "Unknown"
    if v < 1:       return "Very_Low"
    if v < 3:       return "Low"
    if v < 6:       return "Medium"
    return "High"

df_clean["Visibility_Level"] = df_clean["Visibility(mi)"].apply(classify_visibility)

# 2.4 Mức nhiệt độ
def classify_temp(t) -> str:
    if pd.isna(t):  return "Unknown"
    if t < 32:      return "Freezing"
    if t < 50:      return "Cold"
    if t < 70:      return "Mild"
    return "Hot"

df_clean["Temp_Level"] = df_clean["Temperature(F)"].apply(classify_temp)

# 2.5 Đánh dấu thời tiết xấu
adverse_groups = {"Rain", "Snow", "Fog", "Storm"}
df_clean["Is_Adverse_Weather"] = df_clean["Weather_Group"].isin(adverse_groups)

# 2.6 Đánh dấu tầm nhìn thấp
df_clean["Is_Low_Visibility"] = df_clean["Visibility(mi)"] < 1.0

# ---------- 3. Điểm quan tâm trên đường (POI) — ép kiểu bool ----------
poi_cols = [
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction",
    "No_Exit", "Railway", "Roundabout", "Station", "Stop",
    "Traffic_Calming", "Traffic_Signal", "Turning_Loop",
]
poi_cols_existing = [c for c in poi_cols if c in df_clean.columns]
for col in poi_cols_existing:
    df_clean[col] = df_clean[col].astype(bool)

# ---------- 4. Kiểm tra kết quả ----------
weather_feature_cols = [
    "Is_Daytime", "Is_Precipitation", "Weather_Group",
    "Visibility_Level", "Temp_Level",
    "Is_Adverse_Weather", "Is_Low_Visibility",
]

print("=" * 70)
print("CÁC ĐẶC TRƯNG THỜI TIẾT ĐÃ TẠO")
print("=" * 70)
for col in weather_feature_cols:
    if col in df_clean.columns:
        print(f"  - {col:22s} | dtype={str(df_clean[col].dtype):10s} | "
              f"n_unique={df_clean[col].nunique():3d} | "
              f"missing={df_clean[col].isna().sum():,}")

print("\n" + "=" * 70)
print("CÁC ĐẶC TRƯNG POI ĐÃ ÉP KIỂU BOOL")
print("=" * 70)
for col in poi_cols_existing:
    print(f"  - {col:20s} | True={df_clean[col].sum():>10,} "
          f"({df_clean[col].mean()*100:5.2f}%)")

# Assert không có missing trong các đặc trưng mới
for col in weather_feature_cols:
    if col in df_clean.columns:
        assert df_clean[col].isna().sum() == 0, f"Cột {col} còn NaN."

# Assert miền giá trị hợp lệ
assert set(df_clean["Weather_Group"].unique()) <= \
    {"Clear", "Cloudy", "Rain", "Snow", "Fog", "Storm", "Other"}, \
    "Weather_Group có giá trị lạ."
assert set(df_clean["Visibility_Level"].unique()) <= \
    {"Very_Low", "Low", "Medium", "High", "Unknown"}, \
    "Visibility_Level có giá trị lạ."
assert set(df_clean["Temp_Level"].unique()) <= \
    {"Freezing", "Cold", "Mild", "Hot", "Unknown"}, \
    "Temp_Level có giá trị lạ."
assert df_clean["Is_Daytime"].dtype == bool, "Is_Daytime phải là kiểu bool."

print(f"\nTổng số cột hiện tại: {len(df_clean.columns)}")

# ---------- 5. Xem mẫu 5 dòng ----------
preview_cols = [
    "Start_Time", "Weather_Condition", "Weather_Group",
    "Is_Precipitation", "Visibility(mi)", "Visibility_Level",
    "Temperature(F)", "Temp_Level",
    "Is_Adverse_Weather", "Is_Low_Visibility", "Is_Daytime",
]
preview_cols = [c for c in preview_cols if c in df_clean.columns]
df_clean[preview_cols].head()

CÁC ĐẶC TRƯNG THỜI TIẾT ĐÃ TẠO
  - Is_Daytime             | dtype=bool       | n_unique=  2 | missing=0
  - Is_Precipitation       | dtype=bool       | n_unique=  2 | missing=0
  - Weather_Group          | dtype=str        | n_unique=  7 | missing=0
  - Visibility_Level       | dtype=str        | n_unique=  4 | missing=0
  - Temp_Level             | dtype=str        | n_unique=  4 | missing=0
  - Is_Adverse_Weather     | dtype=bool       | n_unique=  2 | missing=0
  - Is_Low_Visibility      | dtype=bool       | n_unique=  2 | missing=0

CÁC ĐẶC TRƯNG POI ĐÃ ÉP KIỂU BOOL
  - Amenity              | True=    35,257 ( 1.19%)
  - Crossing             | True=   290,870 ( 9.78%)
  - Junction             | True=   186,218 ( 6.26%)
  - Railway              | True=    22,395 ( 0.75%)
  - Stop                 | True=    81,224 ( 2.73%)
  - Traffic_Signal       | True=   285,659 ( 9.61%)

Tổng số cột hiện tại: 36


,Start_Time,Weather_Condition,Weather_Group,Is_Precipitation,Visibility(mi),Visibility_Level,Temperature(F),Temp_Level,Is_Adverse_Weather,Is_Low_Visibility,Is_Daytime
0,2021-03-31 23:37:00,Fair,Clear,False,10.0,High,61.0,Mild,False,False,False
1,2021-03-31 23:37:00,Fair,Clear,False,10.0,High,61.0,Mild,False,False,False
2,2021-03-31 23:37:30,Fair,Clear,False,10.0,High,66.0,Mild,False,False,False
3,2021-03-31 23:37:30,Fair,Clear,False,10.0,High,66.0,Mild,False,False,False
4,2021-03-31 23:37:30,Fair,Clear,False,10.0,High,60.0,Mild,False,False,False


## 6. Chia tập Train/Test — Time-Series Split

**Vì sao KHÔNG dùng `train_test_split` ngẫu nhiên?**
Đây là dữ liệu có tính **chuỗi thời gian** (time-series): các đặc trưng như điều kiện thời tiết,
mật độ giao thông, hoặc xu hướng theo mùa có tính tự tương quan theo thời gian. Nếu chia ngẫu nhiên,
các bản ghi ở tập test có thể "rò rỉ" thông tin từ các thời điểm liền kề trong tập train (data leakage),
khiến mô hình đánh giá overly-optimistic và không phản ánh đúng khả năng dự báo cho dữ liệu **tương lai** thực tế.

**Cách làm:** sắp xếp dữ liệu theo `Start_Time`, sau đó cắt theo một mốc thời gian —
toàn bộ dữ liệu **trước** mốc đó vào tập train, toàn bộ **sau** mốc đó vào tập test.
Tỉ lệ mặc định: 80% train (giai đoạn đầu) / 20% test (giai đoạn cuối, gần nhất).


In [17]:
TEST_SIZE_RATIO = 0.20

df_clean.sort_values("Start_Time", inplace=True)
df_clean.reset_index(drop=True, inplace=True)

# 80% giai đoạn đầu cho train, 20% giai đoạn cuối cho test.
split_idx = int(len(df_clean) * (1 - TEST_SIZE_RATIO))
split_idx = max(1, min(split_idx, len(df_clean) - 1))

split_date = df_clean.loc[split_idx, "Start_Time"]

train_df = df_clean.iloc[:split_idx].copy()
test_df = df_clean.iloc[split_idx:].copy()

print(f"Moc chia (split date) : {split_date}")
print(
    f"Train set : {train_df.shape[0]:,} dong "
    f"({train_df['Start_Time'].min()} -> {train_df['Start_Time'].max()})"
)
print(
    f"Test set  : {test_df.shape[0]:,} dong "
    f"({test_df['Start_Time'].min()} -> {test_df['Start_Time'].max()})"
)

# Kiểm tra không có chồng lấn thời gian giữa train và test.
assert train_df["Start_Time"].max() <= test_df["Start_Time"].min(),     "Phat hien chong lan thoi gian giua train/test!"

print("\n[OK] Khong co data leakage ve mat thoi gian giua train va test.")


Moc chia (split date) : 2022-10-04 14:55:00
Train set : 2,378,312 dong (2021-03-31 23:37:00 -> 2022-10-04 14:54:50)
Test set  : 594,578 dong (2022-10-04 14:55:00 -> 2023-03-31 23:30:00)

[OK] Khong co data leakage ve mat thoi gian giua train va test.


## 7. Xuất dữ liệu đã làm sạch

Lưu file cho Member 3 (visualization) và Member 4 (modeling). Các file trong `data/`
nên được thêm vào `.gitignore` nếu kích thước lớn — xem hướng dẫn ở mục 8.


## Vi sao dung CSV thay vi TXT?

US Accidents la du lieu dang bang. CSV giu ro cau truc hang/cot va doc truc tiep duoc bang pandas, Excel, Power BI, Tableau va cac cong cu Machine Learning. TXT chi la van ban thuan tuy, nen khong thuan tien bang CSV cho quy trinh nay.

**Ket luan:** giu `accidents_clean_2y.csv`, `train.csv`, `test.csv`. Chi dung TXT neu giang vien hoac he thong cu the yeu cau.

In [18]:
# ============================================================
# XUAT DU LIEU SAU KHI CLEANING
# ============================================================
# accidents_clean_2y.csv : toan bo du lieu da lam sach, loc 2 nam
# gan nhat va da tao cac feature thoi gian.
# train.csv             : 80% du lieu dau theo thu tu thoi gian.
# test.csv              : 20% du lieu cuoi theo thu tu thoi gian.
#
# CSV phu hop hon TXT vi day la du lieu dang bang. CSV co the
# doc truc tiep bang pandas, Excel, Power BI, Tableau va cac
# cong cu Machine Learning.
# ============================================================

from pathlib import Path

# Tao thu muc data neu chua ton tai.
# ../data nghia la thu muc data nam cung cap voi thu muc notebooks.
OUTPUT_DIR = Path("../data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dat ten 3 file dau ra.
clean_path = OUTPUT_DIR / "accidents_clean_2y.csv"
train_path = OUTPUT_DIR / "train.csv"
test_path = OUTPUT_DIR / "test.csv"

# ------------------------------------------------------------
# 1. Luu du lieu da CLEANING
# ------------------------------------------------------------
# utf-8-sig giup Excel tren Windows doc Unicode tot hon.
df_clean.to_csv(clean_path, index=False, encoding="utf-8-sig")

print("Da tao file CLEANING:")
print(f"  {clean_path}")
print(f"  So dong: {len(df_clean):,}")
print(f"  So cot : {len(df_clean.columns):,}")

# ------------------------------------------------------------
# 2. Luu tap TRAIN
# ------------------------------------------------------------
train_df.to_csv(train_path, index=False, encoding="utf-8-sig")

print("\nDa tao file TRAIN:")
print(f"  {train_path}")
print(f"  So dong: {len(train_df):,}")
print(f"  So cot : {len(train_df.columns):,}")

# ------------------------------------------------------------
# 3. Luu tap TEST
# ------------------------------------------------------------
test_df.to_csv(test_path, index=False, encoding="utf-8-sig")

print("\nDa tao file TEST:")
print(f"  {test_path}")
print(f"  So dong: {len(test_df):,}")
print(f"  So cot : {len(test_df.columns):,}")

# ------------------------------------------------------------
# 4. Kiem tra file co thuc su duoc tao hay khong
# ------------------------------------------------------------
print("\n===== KIEM TRA FILE DAU RA =====")

for file_path, file_name, data in [
    (clean_path, "accidents_clean_2y.csv", df_clean),
    (train_path, "train.csv", train_df),
    (test_path, "test.csv", test_df),
]:
    if file_path.exists():
        size_mb = file_path.stat().st_size / (1024 * 1024)
        print(f"[OK] {file_name}")
        print(f"     Kich thuoc: {size_mb:.2f} MB")
        print(f"     Shape: {data.shape}")
    else:
        print(f"[ERROR] Khong tao duoc {file_name}")

# Kiem tra nhanh so o con thieu trong du lieu da cleaning.
remaining_missing = int(df_clean.isna().sum().sum())
print(f"\nTong so o con thieu trong CLEANING: {remaining_missing:,}")

if remaining_missing == 0:
    print("[OK] Du lieu cleaning khong con gia tri thieu.")
else:
    print("[WARNING] Du lieu van con gia tri thieu. Hay kiem tra lai buoc missing.")


Da tao file CLEANING:
  ..\data\accidents_clean_2y.csv
  So dong: 2,972,890
  So cot : 36

Da tao file TRAIN:
  ..\data\train.csv
  So dong: 2,378,312
  So cot : 36

Da tao file TEST:
  ..\data\test.csv
  So dong: 594,578
  So cot : 36

===== KIEM TRA FILE DAU RA =====
[OK] accidents_clean_2y.csv
     Kich thuoc: 702.94 MB
     Shape: (2972890, 36)
[OK] train.csv
     Kich thuoc: 562.57 MB
     Shape: (2378312, 36)
[OK] test.csv
     Kich thuoc: 140.36 MB
     Shape: (594578, 36)

Tong so o con thieu trong CLEANING: 0
[OK] Du lieu cleaning khong con gia tri thieu.


## 8. Ghi chú Git — không commit dữ liệu thô/lớn

Vì notebook đã dùng **KaggleHub** để tải dataset tự động, không cần commit file CSV lớn vào Git.

Có thể thêm các dòng sau vào `.gitignore` ở gốc repository:

```text
# Du lieu tho va du lieu da xu ly (qua lon cho Git)
data/raw/*
data/*.csv
!data/.gitkeep
```

Khi thành viên khác chạy notebook, chỉ cần cài `kagglehub` và có quyền truy cập dataset Kaggle. Notebook sẽ tự tải dữ liệu bằng:

`kagglehub.dataset_download("sobhanmoosavi/us-accidents")`


## 9. Data Dictionary — phục vụ viết phần "Dữ liệu" trong báo cáo Word

Bảng tóm tắt các biến chính sau khi làm sạch, dùng để trích dẫn khi viết mục (2) Dữ liệu:

| Cột | Kiểu dữ liệu | Mô tả | Xử lý |
|---|---|---|---|
| `Start_Time` | datetime | Thời điểm xảy ra tai nạn | Parse datetime, loại dòng lỗi |
| `Severity` | int (1-4) | Mức độ nghiêm trọng của tai nạn | Giữ nguyên |
| `Start_Lat`, `Start_Lng` | float | Tọa độ vị trí tai nạn | Giữ nguyên |
| `City`, `County`, `State` | category | Vị trí hành chính | Impute mode nếu thiếu ít |
| `Weather_Condition` | category | Điều kiện thời tiết lúc xảy ra | Impute mode |
| `Visibility(mi)`, `Temperature(F)`, `Humidity(%)`, `Precipitation(in)` | float | Các chỉ số thời tiết | Impute median |
| `Hour`, `DayOfWeek`, `Month`, `Season`, `Year` | derived | Đặc trưng thời gian trích xuất từ `Start_Time` | Feature engineering |

Chạy cell dưới để tự động sinh bảng thống kê mô tả (mean/median/min/max/tỉ lệ null-trước-xử-lý)
cho phần phụ lục báo cáo.


In [19]:
summary = pd.DataFrame({
    "dtype": df_clean.dtypes.astype(str),
    "n_unique": df_clean.nunique(),
    "original_missing_pct": (missing_pct.reindex(df_clean.columns).fillna(0) * 100).round(2),
})
summary


,dtype,n_unique,original_missing_pct
ID,str,2972890,0.00
Severity,int64,4,0.00
Start_Time,datetime64[ns],1823622,0.00
End_Time,datetime64[ns],2280349,0.00
Start_Lat,float64,1257328,0.00
Start_Lng,float64,1276538,0.00
City,str,10124,0.00
County,str,1628,0.00
State,str,49,0.00
Zipcode,str,447720,0.02


## 10. Kiểm tra cuối pipeline

Kiểm tra nhanh trước khi bàn giao dữ liệu cho Member 3 và Member 4.

In [20]:
print("===== FINAL CHECK =====")
print(f"Du lieu sach    : {df_clean.shape}")
print(f"Train           : {train_df.shape}")
print(f"Test            : {test_df.shape}")
print(f"Start_Time      : {df_clean['Start_Time'].dtype}")
print(f"Thoi gian       : {df_clean['Start_Time'].min()} -> {df_clean['Start_Time'].max()}")
print(f"Missing con lai : {int(df_clean.isna().sum().sum()):,}")

assert len(df_clean) > 0
assert len(train_df) > 0 and len(test_df) > 0
assert train_df["Start_Time"].max() <= test_df["Start_Time"].min()
assert all(
    col in df_clean.columns
    for col in ["Start_Time", "Severity", "Start_Lat", "Start_Lng",
                "Hour", "Month", "Season", "Year"]
)

print("[OK] Pipeline hoan tat: KaggleHub -> Cleaning -> Feature Engineering -> Time-Series Split -> Export.")


===== FINAL CHECK =====
Du lieu sach    : (2972890, 36)
Train           : (2378312, 36)
Test            : (594578, 36)
Start_Time      : datetime64[ns]
Thoi gian       : 2021-03-31 23:37:00 -> 2023-03-31 23:30:00
Missing con lai : 0
[OK] Pipeline hoan tat: KaggleHub -> Cleaning -> Feature Engineering -> Time-Series Split -> Export.
